### Welcome!
<style>
div.text_cell_render {
    background-color: transparent !important;
    border: none !important;
    padding: 0px !important;
}
</style>

In [ ]:
%%time

import warnings
import cartopy.crs as ccrs
import geoviews.feature as gf
import holoviews as hv
import hvplot.xarray
from holoviews import opts
import uxarray as ux
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
hv.extension("bokeh")
# hv.extension("matplotlib")

import geoviews as gv
import cartopy.crs as ccrs
import geopandas as gp
import os, sys
sys.path.append(os.path.join(os.getcwd(), "..")) 

# bkg_map = gf.coastline(projection=ccrs.PlateCarree(), line_width=1, scale="50m") \
#          * gf.states(projection=ccrs.PlateCarree(), line_width=1, line_color='gray', scale="50m")
coast_lines = gf.coastline(projection=ccrs.PlateCarree(), line_width=1, scale="50m")
state_lines = gf.states(projection=ccrs.PlateCarree(), line_width=1, line_color='gray', scale="50m")
counties_gdf = gp.read_file("../data/shapes_county/cb_2018_us_county_500k.shp")  # https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_county_500k.zip
county_lines = gv.Polygons(counties_gdf, crs=ccrs.PlateCarree()).opts(fill_alpha=0, line_color='gray', line_width=0.5)
# county_lines # show the county_lines plot. This step is slow as we will plot many polygons

In [ ]:
def mpas_hslice(ux_hslice, title, minval=None, maxval=None, width=800, height=500):
    import math
    # Get min and max
    if minval is None:
        amin = ux_hslice.min().item()
        minval = math.floor(amin)
    if maxval is None:
        amax = ux_hslice.max().item()
        maxval = math.ceil(amax)
    title += f" min={amin:.1f} max={amax:.1f}"
        
    # generate contour plot     
    contour_plot = hv.operation.contours(
        ux_hslice.plot(),
        levels=np.linspace(minval, maxval, num=20),  # levels=np.arange(minval, maxval, 0.5)
        filled=True
    ).opts(line_color=None)  # line_width=0.001

    # overlay with background bkg_map
    final = contour_plot.opts(
        width=width, height=height,
        cmap='coolwarm', clim=(minval, maxval), colorbar=True,  # cmap="inferno"
        show_legend=False, tools=['hover'], title=title
    )
    return final

In [ ]:
def mpas_vslice_along_lon(uxvar, lon, lat1=None, lat2=None, lat_incr=None, minval=None, maxval=None, width=600, height=530, clevels=20):
    import math
    ux_vslice = uxvar.isel(Time=0).cross_section.constant_longitude(lon)
    del ux_vslice.uxgrid._ds['edge_node_connectivity'] # temporary uxarray bug and it will be fixed soon
    del ux_vslice.uxgrid._ds['edge_lon']
    del ux_vslice.uxgrid._ds['face_lon']

    if lat1 is None:        
        print("min_lat: ", ux_vslice.uxgrid.node_lat.min().item())
        print("max_lat: ", ux_vslice.uxgrid.node_lat.max().item())
        return
        
    # create latitude bins 
    lats = np.arange(lat1, lat2, lat_incr)
    
    # remap faces
    face_indices = []
    for lat in lats:
        face_idx = ux_vslice.uxgrid.get_faces_containing_point(point_lonlat = np.array([lon, lat]))
        face_indices.append(face_idx)
    
    face_indices = np.array(face_indices).squeeze()
    face_DataArray = xr.DataArray(data=np.array(face_indices), dims=['lat'])
    
    ux_vslice_selected = ux_vslice.isel(n_face=face_DataArray, ignore_grid=True)
    # Get min and max
    if minval is None:
        amin = ux_vslice_selected.min().item()
        minval = math.floor(amin)
    if maxval is None:
        amax = ux_vslice.max().item()
        maxval = math.ceil(amax)
    title = f"constant_lon={lon} min={amin:.1f} max={amax:.1f}"
    levels = np.linspace(minval, maxval, num=clevels)
    
    # ux_vslice_selected.to_xarray().transpose().plot.contourf()  # plot using matplotlib
    # plt.title(f"lon = {lon}")  # add a title to matplotlib
    return ux_vslice_selected.to_xarray().transpose().hvplot.contourf(levels=levels, width=width, height=height, title=title)  # aspect=1

In [ ]:
def mpas_vslice_along_lon2(uxvar, lon, minval=None, maxval=None, width=600, height=530, clevels=20):
    import math
    ux_vslice = uxvar.isel(Time=0).cross_section.constant_longitude(lon)
    del ux_vslice.uxgrid._ds['edge_node_connectivity'] # temporary uxarray bug and it will be fixed soon
    del ux_vslice.uxgrid._ds['edge_lon']
    del ux_vslice.uxgrid._ds['face_lon']
       
    # sort lats
    sort_indices = ux_vslice.uxgrid.face_lat.argsort()
    sorted_lons = ux_vslice.uxgrid.face_lon[sort_indices]
    sorted_lats = ux_vslice.uxgrid.face_lat[sort_indices]
    
    # remap faces
    face_indices = []
    for mylon, mylat in zip(sorted_lons, sorted_lats):
        face_idx = ux_vslice.uxgrid.get_faces_containing_point(point_lonlat = np.array([mylon.item(), mylat.item()]))
        face_indices.append(face_idx)
    
    face_indices = np.array(face_indices).squeeze()
    face_DataArray = xr.DataArray(data=np.array(face_indices), dims=['lat'])
    
    ux_vslice_selected = ux_vslice.isel(n_face=face_DataArray, ignore_grid=True)
    # Get min and max
    if minval is None:
        amin = ux_vslice_selected.min().item()
        minval = math.floor(amin)
    if maxval is None:
        amax = ux_vslice.max().item()
        maxval = math.ceil(amax)
    title = f"constant_lon={lon} min={amin:.1f} max={amax:.1f}"
    levels = np.linspace(minval, maxval, num=clevels)
    
    # ux_vslice_selected.to_xarray().transpose().plot.contourf()  # plot using matplotlib
    # plt.title(f"lon = {lon}")  # add a title to matplotlib

    # return the slice array with a lat dim
    # ux_vslice_selected = ux_vslice_selected.assign_coords(lats=xr.DataArray(data=sorted_lats, dims=['lat']))
    # return ux_vslice_selected.to_xarray().transpose() # return the slice array
    
    return ux_vslice_selected.to_xarray().transpose().hvplot.contourf(levels=levels, width=width, height=height, title=title)  # aspect=1

In [ ]:
tmp = mpas_vslice_along_lon2(uxvar, lon=-85.77, clevels=10)
tmp

In [ ]:
def mpas_vslice_along_lat(uxvar, lat, lon1=None, lon2=None, lon_incr=None, minval=None, maxval=None, width=600, height=530, clevels=20):
    import math
    ux_vslice = uxvar.isel(Time=0).cross_section.constant_latitude(lat)
    del ux_vslice.uxgrid._ds['edge_node_connectivity'] # temporary uxarray bug and it will be fixed soon
    del ux_vslice.uxgrid._ds['edge_lon']
    del ux_vslice.uxgrid._ds['face_lon']

    if lon1 is None:        
        print("min_lon: ", ux_vslice.uxgrid.node_lon.min().item())
        print("max_lon: ", ux_vslice.uxgrid.node_lon.max().item())
        return
        
    # create latitude bins 
    lons = np.arange(lon1, lon2, lon_incr)
    
    # remap faces
    face_indices = []
    for lon in lons:
        face_idx = ux_vslice.uxgrid.get_faces_containing_point(point_lonlat = np.array([lon, lat])) #[0]
        face_indices.append(face_idx)
    
    face_indices = np.array(face_indices).squeeze()
    face_DataArray = xr.DataArray(data=np.array(face_indices), dims=['lon'])

    ux_vslice_selected = ux_vslice.isel(n_face=face_DataArray, ignore_grid=True)
    # Get min and max
    if minval is None:
        amin = ux_vslice_selected.min().item()
        minval = math.floor(amin)
    if maxval is None:
        amax = ux_vslice.max().item()
        maxval = math.ceil(amax)
    title = f"constant_lat={lat} min={amin:.1f} max={amax:.1f}"
    levels = np.linspace(minval, maxval, num=clevels)
    
    # ux_vslice_selected.to_xarray().transpose().plot.contourf()  # plot using matplotlib
    # plt.title(f"lat = {lat}")  # add a title to matplotlib
    return ux_vslice_selected.to_xarray().transpose().hvplot.contourf(levels=levels, width=width, height=height, title=title)  # aspect=1

## read MPAS model data

In [ ]:
%%time

grid_file="../data/samples/mpasjedi/invariant.nc"
bkg_file="../data/samples/mpasjedi/bkg.nc"
ana_file="../data/samples/mpasjedi/ana.nc"

uxds_a = ux.open_dataset(grid_file, ana_file)
uxds_b = ux.open_dataset(grid_file, bkg_file)
# uxds_a   # display information about the dataset

## plot a horizontal cross section

In [ ]:
%%time

uxvar0 = uxds_a['theta'] - 273.15

### subset to a small region
lon_center = -105.03
lat_center = 39.0
lon_incr = 5 # degree
lat_incr = 3 # degree
lon_bounds = (lon_center - lon_incr, lon_center + lon_incr)
lat_bounds = (lat_center - lat_incr, lat_center + lat_incr)
uxvar1 = uxvar0.subset.bounding_box(lon_bounds, lat_bounds,)

# use the whole domain or subdomain
uxvar = uxvar0  # uxvar0 

# generate color plot
nt = 0  # time dimension
lev = 0  # vertical level
plot = mpas_hslice(uxvar.isel(Time=nt, nVertLevels=lev), title=f'lev={lev}', width=800, height=550)  # for subdomain
# plot = mpas_hslice(uxvar.isel(Time=nt, nVertLevels=lev), bkg_map, title=f'lev={lev}')  # for the whole domain
# hv.save(plot, 'theta.png')  # save a local png file along with displaying it
#(plot * counties).opts(xlim=(-110, -100), ylim=(36, 42))
plot * coast_lines * state_lines

## plot variable differences:

In [ ]:
%%time

var_name = "theta"
uxdiff0 = uxds_a[var_name] - uxds_b[var_name]

### subset to a small domain
lon_center = -105.03
lat_center = 39.0
lon_incr = 5 # degree
lat_incr = 3 # degree
lon_bounds = (lon_center - lon_incr, lon_center + lon_incr)
lat_bounds = (lat_center - lat_incr, lat_center + lat_incr)
uxdiff1 = uxdiff0.subset.bounding_box(lon_bounds, lat_bounds,)

# choose to use the whole domain or the subdomain
uxvar = uxdiff0  # uxdiff1

nt=0  # time dimension
plot_levels = [0, 29, 42]  # [0, 19, 29, 39, 49, 58]

plots = []
for lev in plot_levels:
    tmp = mpas_hslice(uxvar.isel(Time=nt, nVertLevels=lev), title=f'lev={lev}')  # for the whole domain
    # tmp = mpas_hslice(uxvar.isel(Time=nt, nVertLevels=lev), title=f'lev={lev}', width=800, height=700)  # for the subdomain
    plots.append(tmp * coast_lines * state_lines)

# plots share one toolbar, which facilitates doing sync'ed zoom-in/out
# hv.Layout(plots).cols(1)

# each plot has its own toolbar, which facilitates controlling each plot individually
for p in plots:
   display(p)

## plot vertical cross sections

In [ ]:
var_name = "theta"
uxdiff0 = uxds_a[var_name] - uxds_b[var_name]

### subset to a small domain
lon_center = -105.03
lat_center = 39.0
lon_incr = 5 # degree
lat_incr = 3 # degree
lon_bounds = (lon_center - lon_incr, lon_center + lon_incr)
lat_bounds = (lat_center - lat_incr, lat_center + lat_incr)
uxdiff1 = uxdiff0.subset.bounding_box(lon_bounds, lat_bounds,)

# choose to use the whole domain or the subdomain
uxvar = uxdiff0  # uxdiff1
import math

In [ ]:
from DAmonitor.mpas import coast_lines, state_lines, county_lines, hslice_contour, vslice_contour

In [ ]:
# tmp = mpas_vslice_along_lon(uxvar, lon=-85.77, lat1=23, lat2=54, lat_incr=0.5, clevels=10)
# display(tmp)
# tmp = mpas_vslice_along_lon2(uxvar, lon=-85.77, lat1=23, lat2=54, lat_incr=0.5, clevels=10)
# tmp

vslice_contour(uxvar, lon=-85.77, clevels=10)
#display(tmp)
# tmp = mpas_vslice_along_lat(uxvar, lat=42.63, lon1=-129.5, lon2=-65.5, lon_incr=0.5, clevels=10)
# display(tmp)

## testing

In [ ]:
lon = -85.77
ux_vslice = uxvar.isel(Time=0).cross_section.constant_longitude(lon)
ux_vslice

In [ ]:
# all data values at a given longitude 
# def mpas_vslice_along_lat(uxvar, lat):
lon = -85.77
ux_vslice = uxvar.isel(Time=0).cross_section.constant_longitude(lon)
del ux_vslice.uxgrid._ds['edge_node_connectivity'] # temporary uxarray bug and it will be fixed soon
del ux_vslice.uxgrid._ds['edge_lon']
del ux_vslice.uxgrid._ds['face_lon']

# print(ux_vslice.uxgrid.node_lon.max().item())
# print(ux_vslice.uxgrid.node_lon.min().item())
# print(ux_vslice.uxgrid.node_lat.max().item())
# print(ux_vslice.uxgrid.node_lat.min().item())
sort_indices = ux_vslice.uxgrid.face_lat.argsort()
sorted_lat = ux_vslice.uxgrid.face_lat[sort_indices]
sorted_lon = ux_vslice.uxgrid.face_lon[sort_indices]

# remap faces
face_indices = []
for lon, lat in zip(sorted_lon, sorted_lat):
    face_idx = ux_vslice.uxgrid.get_faces_containing_point(point_lonlat = np.array([lon.item(), lat.item()]))
    print(face_idx)
    face_indices.append(face_idx)

In [ ]:
# print(face_indices)
face_indices = np.array(face_indices).squeeze()  # does squeeze remove duplicate values?
# face_indices
face_DataArray = xr.DataArray(data=np.array(face_indices), dims=['lat']) # da = data_array

ux_vslice_selected = ux_vslice.isel(n_face=face_DataArray, ignore_grid=True)
# ux_vslice_selected  # examine the data content
# ux_vslice_selected.to_xarray().transpose().plot.contourf()  # plot using matplotlib
# plt.title(f"lon = {lon_center}")  # add a title to matplotlib
ux_vslice_selected.to_xarray().transpose().hvplot.contourf(levels=np.arange(-16, 16, 2), width=600, height=530)  # aspenct=1

## plot grids

In [ ]:
grid_file="../data/samples/mpasjedi/invariant.nc"
uxgrid = ux.open_grid(grid_file)
uxgrid

In [ ]:
uxgrid.plot(title="Default Plot Function")

In [ ]:
(
    uxgrid.plot.edges(color="black")
    * uxgrid.plot.nodes(marker="o", size=150).relabel("Corner Nodes")
    * uxgrid.plot.face_centers(marker="s", size=150).relabel("Face Centers")
    * uxgrid.plot.edge_centers(marker="^", size=150).relabel("Edge Centers")
).opts(title="Grid Coordinates", legend_position="top_right")

In [ ]:
projection = ccrs.Robinson()

uxvar_lon.plot(
    projection=projection,
    cmap="inferno",
    features=["borders", "coastline"],
    title="Anything",
    width=200,
)